# `Business Case`

## `Requerimientos Funcionales`

* `Añadir` productos al inventario, incluyendo nombre, cantidad (enteros), precio unitario, valor total del inventario.

* `Eliminar` productos del inventario. Evitar el ofrecimiento de productos que ya no son ofrecidos por la organización.

* `Consultar` el estado actual de inventario. Medir la capacidad de oferta, puntos de reabastecimiento, etc.

* `Exportar` informe en formato PDF.

## `Requerimientos Técnicos`

* `Normalización` del nombre del producto (`title()`, `strip()`).

* `Separación` visual con caracteres (`"-" * 30`) para un diseño más organizado.

* `Menú` interactivo para que el usuario elija acciones (`while()`).

* `Representación` del inventario y productos con estructuras de datos (e.g. `list`, `dict`…).

* `Implementación` de la librería `fpdf`.  Para la exportación del informe en formato `PDF`.


In [ ]:
# !pip install fpdf
# !pip install PyPDF2

In [6]:
import os


inventario = []
IVA = 0.19



def mostrar_menu():
    print("\n🛒 Bienvenidos")
    print("1. Agregar Producto")
    print("2. Ver Inventario")
    print("3. Eliminar Producto")
    print("4. Exportar informe inventario actual formato PDF")
    print("5. Salir")


def agregar_producto(lista_inventario):

    """
    Solicita al usuario los datos de un producto (nombre, cantidad y precio)
    y lo agrega como un diccionario a la lista de inventario.
    
    Parametros:
        lista_inventario (list): La lista donde se guardara el producto.
    """
    producto = input("Ingrese el nombre del producto: ").upper().strip()
    cantidad= int(input("Ingrese la cantidad: "))
    precio_unitario = float(input("Precio del producto: "))
                
    valor_total = cantidad * precio_unitario
        
    producto = {
            "nombre":producto,
            "cantidad":cantidad,
            "precio":precio_unitario,
            "valor_total":valor_total
     }
        
    lista_inventario.append(producto)
    print(f'Producto: {producto["nombre"]} agregado correctamente')

def mostrar_inventario(lista_inventario,iva):

    """
    Muestra los productos guardados en el inventario con sus precios
    y calcula el subtotal, el IVA aplicado y el total final.
    
    Parametros:
    lista_inventario (list): La lista de productos a mostrar.
    iva (float): El porcentaje de IVA a aplicar en el calculo.
    """
          
    if not lista_inventario:                     
        print("El inventario esta vacio")            
    else:
        subtotal = 0
        print("\n Productos en Inventario")
                    
        for n_producto, prod in enumerate(lista_inventario, 1):
            print(f'#{n_producto} {prod["nombre"]} | {prod["cantidad"]} | ${prod["precio"]} | {prod["valor_total"]:.2f}')              
            subtotal += prod["valor_total"] 
        
        total_iva = subtotal * iva
        total_final = subtotal + total_iva
        
        print("-" * 40)
        print(f'El valor total es de: ${subtotal:.2f}')
        print(f'El IVA total de esta boleta es de: {total_iva:.2f}')
        print(f'El valor con IVA total es de: {total_final:.2f}')

def eliminar_producto(lista_inventario):
    """
    Busca un producto por su nombre en la lista de inventario 
    y lo elimina si lo encuentra.
    
    Parametros:
        lista_inventario (list): La lista sobre la cual se realizara la busqueda y eliminacion.
    """
    if not lista_inventario:
            print("No hay productos para eliminar")
    else:
        producto_eliminar = input("Nombre del producto a eliminar").title().strip()
        encontrado = False 
        
        for prod in lista_inventario:
            if prod["nombre"] == producto_eliminar: 
                lista_inventario.remove(prod) 
                print(f'Producto {producto_eliminar} eliminado')
                encontrado = True 
                break 
        
        if not encontrado:
            print("Producto no encontrado") 

def generar_informe(lista_inventario,iva):

   from fpdf import FPDF 
   from PyPDF2 import PdfReader,PdfWriter
   import io

   autor = input("Nombre de quien genera el informe: ")
   revisor = input("Nombre de quien revisa el informe: ")

   pdf = FPDF()
   pdf.add_page()

   pdf.ln(36)

   pdf.set_font("helvetica", size=20, style="B")
   pdf.cell(w=190, h=10, txt="Sistema de Control de Inventario", align="C")
   
   pdf.ln(18)


   resumen = """
En seguida se muestra inventario actual para todos los productos despues de su gestión (añadir o eliminar productos).
También se muestra el Total Unificado del inventario en la tabla que se muestra en seguida
"""

   pdf.set_font("helvetica",size=16)
   pdf.multi_cell(w=180,h=10, txt=resumen.strip(),align="J")
   pdf.ln(10)

   ancho_col = [75, 25, 45, 45]

   # Agregar encabezados a la tabla

   encabezado = ["Producto","Cantidad","Precio Unitario","Valor Total"]

   for i,nombre_col in enumerate(encabezado):
       pdf.set_font("helvetica", size=10,style="B")
       pdf.cell(w= ancho_col[i],h=10,txt=nombre_col,border=1,align="C")

   pdf.ln()

   # Agregar data en las fila
   producto = [list(producto.values()) for producto in lista_inventario]

   for fila in producto:
       for i , item in enumerate(fila):
           if i == 0 and len(str(item)) > 20:
               pdf.set_font("helvetica",size=8)
           else:
               pdf.set_font("helvetica",size=10)

           if (isinstance(item,float) or isinstance(item,int)) and i != 1:
               pdf.cell(w=ancho_col[i],h=8,txt=f'${item:.2f}',border=1,align="C")
           else:
               pdf.cell(w=ancho_col[i],h=8,txt=str(item),border=1,align="C")
       pdf.ln()
               
   subtotal = sum(prod["valor_total"] for prod in lista_inventario)
   iva_total = subtotal * iva
   gran_total = subtotal + iva_total

   ancho_etiqueta = sum(ancho_col[:-1])
   ancho_valor = ancho_col[-1]

   pdf.set_font("helvetica",size=12,style="B")

   pdf.cell(w=ancho_etiqueta, h=8, txt="Subtotal:", border=1,align="R")
   pdf.cell(w=ancho_valor,h=8,txt=f'${subtotal:.2f}',border=1,align="C")
   pdf.ln()

   pdf.cell(w=ancho_etiqueta,h=8,txt=f'IVA({int(iva*100)}%)',border=1,align="R")
   pdf.cell(w=ancho_valor,h=8,txt=f'${iva_total:.2f}',border=1,align="C")
   pdf.ln()

   pdf.cell(w=ancho_etiqueta, h=8, txt="Total Unificado:", border=1,align="R")
   pdf.cell(w=ancho_valor,h=8, txt=f'${gran_total:.2f}',border=1,align="C")

   pdf.ln(30)
   pdf.set_font("helvetica",size=11)

   y_linea = pdf.get_y()

   pdf.line(15,y_linea,85,y_linea)
   pdf.line(115,y_linea,185,y_linea)

   pdf.ln(3)

   ancho_firma = 70
   espacio_central = 30
   margen_izq = 15

   pdf.set_x(margen_izq)

   pdf.cell(w=ancho_firma,h=8,txt= f'Autor: {autor.strip().title()}',border="C")
   pdf.cell(w=espacio_central,h=8,txt="",border=0)
   pdf.cell(w=ancho_firma,h=8,txt= f'Revisor: {revisor.strip().title()}',border="C")

   buffer = io.BytesIO()
   pdf_bytes = pdf.output(dest="S").encode("latin1")
   buffer.write(pdf_bytes)
   buffer.seek(0)

   # Leer plantilla
   template_pdf = PdfReader("plantilla.pdf")
   overlay_pdf = PdfReader(buffer)
   writer = PdfWriter()

   # Combinar páginas
   template_page = template_pdf.pages[0]
   overlay_page = overlay_pdf.pages[0]
   template_page.merge_page(overlay_page)
   writer.add_page(template_page)


   # Generar archivo PDF
   with open("Informe Sistema de Gestion de Inventario.pdf","wb") as f:
       writer.write(f)

   print("\n ✅ Archivo PDF generado con exito!!!")

    
while True:
    mostrar_menu()
    opcion = input("Elija una opcion (1-5): ").strip()

    """ 
        Freno de mano, esperando la respuesta de entrada del usuario

        En consola se verá limpio: Elija una opción (1-4): 1
    
        Si el usuario escribe en la entrada del input opcion "1" lo dirige a la funcion correspondiente
    
    """

    if opcion == "1":
        agregar_producto(inventario)
    elif opcion == "2":
        mostrar_inventario(inventario, IVA)
    elif opcion == "3":
        eliminar_producto(inventario)
    elif opcion == "4":
        generar_informe(inventario,IVA)
    elif opcion == "5":
        print("\n¡Hasta luego! Nos vemos pronto...")
        break
    else:
        print("\nOpción no válida. Elija una opción de 1 a 4.")





🛒 Bienvenidos
1. Agregar Producto
2. Ver Inventario
3. Eliminar Producto
4. Exportar informe inventario actual formato PDF
5. Salir
Producto: MOUSE INALAMBRICO agregado correctamente

🛒 Bienvenidos
1. Agregar Producto
2. Ver Inventario
3. Eliminar Producto
4. Exportar informe inventario actual formato PDF
5. Salir
Producto: TECLADO MECANICO agregado correctamente

🛒 Bienvenidos
1. Agregar Producto
2. Ver Inventario
3. Eliminar Producto
4. Exportar informe inventario actual formato PDF
5. Salir
Producto: MONITOR 27' agregado correctamente

🛒 Bienvenidos
1. Agregar Producto
2. Ver Inventario
3. Eliminar Producto
4. Exportar informe inventario actual formato PDF
5. Salir
Producto: MOUSE PAD GRANDE agregado correctamente

🛒 Bienvenidos
1. Agregar Producto
2. Ver Inventario
3. Eliminar Producto
4. Exportar informe inventario actual formato PDF
5. Salir

 ✅ Archivo PDF generado con exito!!!

🛒 Bienvenidos
1. Agregar Producto
2. Ver Inventario
3. Eliminar Producto
4. Exportar informe inventa